# Lesson 10: Cameras and Vision

You've learned how your camera captures images and how detectors interpret them on the lesson webpage. Today, you'll test those ideas with real classroom objects.

## Get ready

- Keep your car stationary on a stable surface with the motors stopped.
- Gather a brightly colored object, a second object of the same color, and a plain background.
- Use an instructor-approved scene. Do not capture people without permission.
- Finish other driving and vision activities before starting this notebook. Ask your instructor if a detector is already running.
- Select the **Robot Car** kernel. Run cells individually, and reposition objects only between observations.

## 1. Prepare your tools

Run this setup cell first. Set `RUN_CAMERA` to `True` only when your testing area is ready. Pictures use unique names so repeating an experiment will not normally replace earlier work.

`take_picture()` opens a car session, takes a picture, and closes the session. Use the path returned by `capture()` to see the saved image.

In [ ]:
import time
import math
from uuid import uuid4
from IPython.display import Image, display

RUN_CAMERA = False  # Set True when you are ready and have permission.


def take_picture(label):
    from betabox_robotics import BetaboxCar

    filename = f"lesson10_{label}_{uuid4().hex[:12]}.jpg"
    car = BetaboxCar()
    try:
        car.stop()
        picture = car.capture(filename=filename)
        print("Saved:", picture.path)
        return picture.path
    finally:
        car.close()

## 2. Take your first picture

Place your colorful object in front of the camera, wait for the view to settle, and run the next cell. Look for the printed path and the image below the cell. You can also open the file in `~/media/pictures/`.

In [ ]:
if RUN_CAMERA:
    first_picture = take_picture("first_picture")
    display(Image(filename=str(first_picture)))
else:
    print("Camera activity disabled.")

## 3. Change the scene

Use the same object for the next three observations. Run one cell at a time, allowing the view to settle before each capture. Change only the condition named in the instructions.

**Picture 1:** Place the object in the center of the view under normal lighting.

In [ ]:
if RUN_CAMERA:
    normal = take_picture("normal")
    display(Image(filename=str(normal)))

**Picture 2:** Move the object farther away. Keep its position centered and keep the lighting the same.

In [ ]:
if RUN_CAMERA:
    farther = take_picture("farther")
    display(Image(filename=str(farther)))

**Picture 3:** Return the object to its original position. Change the lighting with your instructor's approval, without covering the camera.

In [ ]:
if RUN_CAMERA:
    different_light = take_picture("different_light")
    display(Image(filename=str(different_light)))

Compare your pictures. Record a short observation in each row to help you with the portal reflection later.

| Picture | Condition | What changed in the image? |
|---|---|---|
| 1 | Normal position | |
| 2 | Farther away | |
| 3 | Different lighting | |

## 4. Explore color detection

Run the following helper cell. It enables one detector, waits for a new result, and attempts to disable that detector before closing the car. It won't take over a detector that is already enabled.

A fresh result with an empty detection list means nothing matched in that frame. A timeout, missing result, or detector error is different: the observation was unsuccessful. The helper reports those problems rather than treating them as an empty scene.

In [ ]:
def wait_for_result(car, source, after, timeout=8.0):
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        result = car.metadata(source)
        if (result is not None and result.source == source
                and math.isfinite(result.timestamp)
                and result.timestamp > after):
            if result.data.get("error"):
                raise RuntimeError(str(result.data["error"]))
            return result
        time.sleep(0.2)
    raise TimeoutError("No fresh detection result. Check the camera and detector.")


def observe(source, color="red"):
    from betabox_robotics import BetaboxCar

    car = BetaboxCar()
    attempted_enable = False
    try:
        car.stop()
        status = car.detection_status()
        if source not in status.detectors:
            raise RuntimeError(f"Detector not available: {source}")
        if status.is_enabled(source):
            raise RuntimeError("Detector is already enabled. Ask your instructor before continuing.")
        attempted_enable = True
        if source == "color":
            car.enable_color_detection(color)
        else:
            car.enable_detection(source)
        return wait_for_result(car, source, time.time())
    finally:
        try:
            if attempted_enable:
                if source == "color":
                    car.disable_color_detection()
                else:
                    car.disable_detection(source)
        finally:
            car.close()


def show_colors(result):
    print("Detected regions:", len(result.detections))
    for detection in result.detections:
        print(detection.label, "center:", detection.center,
              "area:", detection.data.get("area"))

### Try it: Find a color

Start with a red object centered in view. If you have a different color, try `blue`, `green`, or `yellow` instead. Run the next cell and inspect the reported label, center coordinates, and area. The results describe regions, not necessarily separate objects.

In [ ]:
color = "red"

if RUN_CAMERA:
    result = observe("color", color)
    show_colors(result)

### Experiment: Move the object

Rerun the previous observation cell after each change. Keep the car still, use the same color, and begin with normal lighting. Compare the x coordinate and area across the results. If more than one region appears, record the region you intended to detect.

| Condition | Regions detected | Center (x, y) | Area |
|---|---|---|---|
| Center | | | |
| Left | | | |
| Right | | | |
| Farther away | | | |
| Two objects of the same color | | | |
| Object removed | | | |

**Your turn:** Choose a different supported color and repeat two observations. Change `color` in the previous code cell and run it again. Note any unexpected detections.

## 5. Explore object detection (if available)

Ask your instructor whether the object detection model is configured. Place a supported classroom object, such as a bottle, in view. Enable this optional cell only with permission.

Rerun it after changing the object's position, distance, or lighting. Compare labels, confidence scores, and bounding boxes. If the model is unavailable or reports an error, tell your instructor and continue with color detection for the final project.

In [ ]:
RUN_OBJECTS = False  # Set True only if the model is available.

if RUN_CAMERA and RUN_OBJECTS:
    result = observe("objects")
    print("Detections:", len(result.detections))
    for detection in result.detections:
        print("Label:", detection.label,
              "Confidence:", detection.confidence,
              "Box:", detection.box,
              "Center:", detection.center)

## 6. Your challenge: Visual Observation Program

Choose an approved colorful object. Build a small program that saves at least three pictures and investigates a color or supported object category.

1. Capture the original position, then move the object sideways, then change its distance.
2. Save each image with a different name and display its path. Use `take_picture()` or your own capture code with `try` and `finally` cleanup.
3. After each scene change, wait for a fresh detection using `observe()` and report the results. You may use `show_colors()` or write your own output for object detections.
4. Inspect your saved pictures, and fill in the observation table below.

The starter uses `input()` so you can reposition the object between captures. Complete the missing instructions before setting `RUN_CHALLENGE` to `True`. If a vision operation fails, stop and investigate before continuing. Keep the motors stopped.

In [ ]:
RUN_CHALLENGE = False

if RUN_CAMERA and RUN_CHALLENGE:
    scenes = ["original", "sideways", "distance"]
    saved_pictures = []
    chosen_color = "red"

    for scene in scenes:
        input(f"Prepare the {scene} scene. Press Enter when ready: ")
        # TODO: Capture a picture with a unique name.
        # TODO: Store its path in saved_pictures.
        # TODO: Get a fresh detector result and display what it found.

    # TODO: Print the paths of every saved picture.

### Compare your results

| Scene | Saved filename | Detection result | Observation |
|---|---|---|---|
| Original position | | | |
| Sideways movement | | | |
| Different distance | | | |

Your project is complete when you have three distinct images, at least three detector observations (including a recorded failure if one occurred), and can demonstrate that the detector has been disabled and the car connection released.

**Extra challenge:** Repeat one scene with different lighting. Compare the saved pictures and the detector's results. Keep the original scene and change only the lighting.

## 7. Finish safely

The provided helpers close each car connection and attempt to disable the detector they enabled. If you created your own car connection, disable any detector you enabled before closing that car object. If cleanup reports an error, ask your instructor to check the vision service before another experiment.

Keep the camera pointed at approved objects. Save your notebook and return to **Unit 2, Lesson 10** in the Betabox web portal to complete your reflection.